# Resolução — S03-05

Notebook em **Julia**, feito para atender aos requisitos pedido pelo professor 
Prof. Kenny Vinente dos Santos, Dr. | kennyvinente@ufam.edu.br
# Nome: Renyer Montefusco levy
# Matrícula: 2240917
Resolução em Julia dos algoritmos pedidos no notebook do professor.


### Pacotes importados

In [1]:
using LinearAlgebra
using Printf

## Chapter 11: Descent methods and line search

Métodos de descida usam uma direção:

\[
d_k
\]

tal que:

\[
\nabla f(x_k)^T d_k < 0
\]

Depois realizamos uma busca linear para encontrar um passo adequado:

\[
x_{k+1}=x_k+\alpha_k d_k
\]


# Algorithm 11.2 — Initialization of the exact line search

O objetivo é encontrar um intervalo inicial contendo um mínimo local.


In [2]:
function initialize_exact_line_search(h, x0; alpha0=0.0, delta=0.1, maxiter=100)

    alpha_prev = alpha0
    h_prev = h(alpha_prev)

    alpha = alpha_prev + delta
    h_alpha = h(alpha)

    historico = []

    push!(historico, (iter=0, alpha=alpha_prev, h=h_prev))
    push!(historico, (iter=1, alpha=alpha, h=h_alpha))

    if h_alpha > h_prev
        return (alpha_prev, alpha), historico
    end

    for k in 1:maxiter

        delta *= 2

        alpha_next = alpha + delta
        h_next = h(alpha_next)

        push!(historico, (iter=k+1, alpha=alpha_next, h=h_next))

        if h_next > h_alpha
            return (alpha_prev, alpha_next), historico
        end

        alpha_prev = alpha
        alpha = alpha_next
        h_alpha = h_next
    end

    error("Intervalo não encontrado.")
end

function mostrar_historico_busca(historico)

    println("Iteração | alpha | h(alpha)")
    println("------------------------------------------------")

    for linha in historico

        @printf("%8d | %.10f | %.16f\n",
            linha.iter,
            linha.alpha,
            linha.h
        )
    end
end

mostrar_historico_busca (generic function with 1 method)

## Teste com Example 11.3

\[
h(x)=(2+x)\cos(2+x)
\]


In [3]:
h(x) = (2 + x) * cos(2 + x)

intervalo, historico = initialize_exact_line_search(h, 0.0)

mostrar_historico_busca(historico)

println()
println("Intervalo encontrado:")
display(intervalo)

Iteração | alpha | h(alpha)
------------------------------------------------
       0 | 0.0000000000 | -0.8322936730942848
       1 | 0.1000000000 | -1.0601768196597010
       2 | 0.3000000000 | -1.5324348489435951
       3 | 0.7000000000 | -2.4409947834460652
       4 | 1.5000000000 | -3.2775984055177871
       5 | 3.1000000000 | 1.9276864878361990

Intervalo encontrado:


(0.7000000000000001, 3.1)

# Algorithm 11.3 — Exact line search: quadratic interpolation

O método usa interpolação quadrática para aproximar o mínimo da função.


In [4]:
function quadratic_interpolation_line_search(h, a, b; eps=1e-10, maxiter=100)

    historico = []

    for k in 1:maxiter

        ha = h(a)
        hb = h(b)

        c = (a + b)/2
        hc = h(c)

        numerador =
            ( (c^2 - b^2)*ha +
              (b^2 - a^2)*hc +
              (a^2 - c^2)*hb )

        denominador =
            ( (c - b)*ha +
              (b - a)*hc +
              (a - c)*hb )

        if abs(denominador) < eps
            break
        end

        x_min = 0.5 * numerador / denominador

        h_min = h(x_min)

        push!(historico, (
            iter=k,
            x=x_min,
            h=h_min
        ))

        if abs(h_min - hc) < eps
            return x_min, historico
        end

        if x_min < c
            b = c
        else
            a = c
        end
    end

    return x_min, historico
end

function mostrar_historico_interpolacao(historico)

    println("Iteração | x | h(x)")
    println("------------------------------------------------")

    for linha in historico

        @printf("%8d | %.12f | %.16f\n",
            linha.iter,
            linha.x,
            linha.h
        )
    end
end

mostrar_historico_interpolacao (generic function with 1 method)

## Teste com Example 11.3

\[
h(x)=(2+x)\cos(2+x)
\]


In [5]:
h(x) = (2 + x) * cos(2 + x)

xmin, historico_interp =
    quadratic_interpolation_line_search(h, -2.0, 2.0)

mostrar_historico_interpolacao(historico_interp)

println()
println("Mínimo aproximado:")
println(xmin)

println("h(xmin) = ", h(xmin))

LoadError: UndefVarError: `x_min` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# Algorithm 11.5 — Line search

Aplicamos busca linear em uma função quadrática:

\[
f(x)=\frac{1}{2}x_1^2+\frac{9}{2}x_2^2
\]


In [6]:
function line_search(f, gradf, x, d; alpha0=1.0, rho=0.5, c=1e-4)

    alpha = alpha0

    fx = f(x)
    g = gradf(x)

    while f(x + alpha*d) > fx + c*alpha*dot(g, d)

        alpha *= rho

        if alpha < 1e-12
            break
        end
    end

    return alpha
end

line_search (generic function with 1 method)

## Example 11.2

\[
f(x)=\frac{1}{2}x_1^2+\frac{9}{2}x_2^2
\]


In [7]:
function f_quad(x)

    x1 = x[1]
    x2 = x[2]

    return 0.5*x1^2 + 4.5*x2^2
end

function grad_quad(x)

    x1 = x[1]
    x2 = x[2]

    return [
        x1,
        9*x2
    ]
end

x0 = [1.0, 1.0]

g0 = grad_quad(x0)

d0 = -g0

alpha = line_search(f_quad, grad_quad, x0, d0)

println("Passo encontrado alpha = ", alpha)

println("Novo ponto:")
display(x0 + alpha*d0)

println("Valor da função no novo ponto:")
println(f_quad(x0 + alpha*d0))

Passo encontrado alpha = 0.125
Novo ponto:


2-element Vector{Float64}:
  0.875
 -0.125

Valor da função no novo ponto:
0.453125


# Algorithm 11.6 — Steepest descent

O método do gradiente descendente usa:

\[
d_k=-\nabla f(x_k)
\]

e busca linear para encontrar \(\alpha_k\).


In [8]:
function steepest_descent(f, gradf, x0; eps=1e-8, maxiter=100)

    x = float.(x0)

    historico = []

    for k in 0:maxiter

        g = gradf(x)

        push!(historico, (
            iter=k,
            x=copy(x),
            norma_gradiente=norm(g),
            valor_funcao=f(x)
        ))

        if norm(g) <= eps
            break
        end

        d = -g

        alpha = line_search(f, gradf, x, d)

        x = x + alpha*d
    end

    return x, historico
end

function mostrar_historico_sd(historico)

    println("Iteração | x | ||grad|| | f(x)")
    println("-------------------------------------------------------------------")

    for linha in historico

        @printf("%8d | [% .8f, % .8f] | %.16e | %.16f\n",
            linha.iter,
            linha.x[1],
            linha.x[2],
            linha.norma_gradiente,
            linha.valor_funcao
        )
    end
end

mostrar_historico_sd (generic function with 1 method)

## Teste na função de Rosenbrock

\[
f(x_1,x_2)=100(x_2-x_1^2)^2+(1-x_1)^2
\]


In [9]:
function rosenbrock(x)

    x1 = x[1]
    x2 = x[2]

    return 100*(x2 - x1^2)^2 + (1 - x1)^2
end

function grad_rosenbrock(x)

    x1 = x[1]
    x2 = x[2]

    return [
        -400*x1*(x2 - x1^2) - 2*(1 - x1),
        200*(x2 - x1^2)
    ]
end

x0_rosen = [-1.2, 1.0]

solucao, historico_sd =
    steepest_descent(
        rosenbrock,
        grad_rosenbrock,
        x0_rosen;
        maxiter=2000
    )

mostrar_historico_sd(historico_sd)

println()
println("Solução aproximada:")
display(solucao)

println("Valor final da função:")
println(rosenbrock(solucao))

println("Norma do gradiente:")
println(norm(grad_rosenbrock(solucao)))

Iteração | x | ||grad|| | f(x)
-------------------------------------------------------------------
       0 | [-1.20000000,  1.00000000] | 2.3286768775422664e+02 | 24.1999999999999957
       1 | [-0.98945312,  1.08593750] | 4.3898520923224993e+01 | 5.1011126637109570
       2 | [-1.06433209,  1.04417187] | 4.5460144022020280e+01 | 5.0470111376546214
       3 | [-1.02345146,  1.06148260] | 3.2789770922880574e+00 | 4.1140390714609625
       4 | [-1.02676510,  1.05600225] | 3.3509138580421074e+00 | 4.1080850212978168
       5 | [-1.02025638,  1.05531644] | 3.4129600330308163e+00 | 4.1021527140761496
       6 | [-1.02383734,  1.04969403] | 3.4655605102780633e+00 | 4.0961281680729726
       7 | [-1.01709245,  1.04912719] | 3.5063757616506526e+00 | 4.0901246018753641
       8 | [-1.02085423,  1.04340448] | 3.5357514479062138e+00 | 4.0840108684295426
       9 | [-1.01396606,  1.04291185] | 3.5522666227421928e+00 | 4.0779179741546772
      10 | [-1.01781085,  1.03713659] | 3.5561039672198205e+

2-element Vector{Float64}:
 0.9886416585405824
 0.9773904100429598

Valor final da função:
0.00012905996478471092
Norma do gradiente:
0.014716764871959939
